In [ ]:
# 1. Install system dependencies and python packages for GRIB processing
!apt-get install libeccodes-dev -qq
!pip install cfgrib -q

print("\nInstallation complete. IMPORTANT: Please go to 'Runtime' -> 'Restart session' for changes to take effect.")

Selecting previously unselected package libeccodes-data.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libeccodes-data_2.24.2-1_all.deb ...
Unpacking libeccodes-data (2.24.2-1) ...
Selecting previously unselected package libeccodes0:amd64.
Preparing to unpack .../libeccodes0_2.24.2-1_amd64.deb ...
Unpacking libeccodes0:amd64 (2.24.2-1) ...
Selecting previously unselected package libeccodes-dev:amd64.
Preparing to unpack .../libeccodes-dev_2.24.2-1_amd64.deb ...
Unpacking libeccodes-dev:amd64 (2.24.2-1) ...
Setting up libeccodes-data (2.24.2-1) ...
Setting up libeccodes0:amd64 (2.24.2-1) ...
Setting up libeccodes-dev:amd64 (2.24.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libhwloc.so.15 is not a symbolic link

/sbin

In [ ]:
#Mounting a new folder from google colab onto drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


This section has informatoin about how to download pressure level variables
and format it


In [ ]:
print('Installing cdsapi...')
!pip install cdsapi -q
print('cdsapi installed successfully!')

import os
from google.colab import userdata

# Retrieve CDS API credentials from Colab secrets

CDSAPI_KEY = userdata.get('CDSAPI_KEY')

if CDSAPI_KEY:
    # Create the .cdsapirc file
    with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
        f.write(f'url: https://cds.climate.copernicus.eu/api')
        f.write(f'key: {CDSAPI_KEY}\n')
    print('CDS API credentials configured.')
else:
    print('Error: CDSAPI_KEY not found in Colab secrets. Please set it up as instructed.')

Installing cdsapi...
cdsapi installed successfully!
CDS API credentials configured.


In [ ]:
import pandas as pd
import os
from datetime import datetime

# --- 0. PATH CONFIGURATION ---
INPUT_CSV = '/content/drive/MyDrive/Shared_data/merged_with_terrain_dataset.csv'
OUTPUT_BASE_DIR = '/content/drive/MyDrive/Pyrocb_data/ERA5_Pressure_Levels'
FINAL_MERGED_CSV = '/content/drive/MyDrive/Pyrocb_data/merged_with_era5_pressure_final.csv'

# 1. Load the input data
df_input = pd.read_csv(INPUT_CSV)

# --- 2. CONFIGURABLE: TARGET IDs ---
# Specify IDs in this list, or set to None to process all rows
TARGET_IDS = ['179','180', '181', '189', '190', '202', '216', '253', '258', '260']

if TARGET_IDS:
    df_to_process = df_input[df_input['pyroCb_id'].astype(str).isin([str(i) for i in TARGET_IDS])].copy()
else:
    df_to_process = df_input.copy()

print(f"Found {len(df_to_process)} specific rows/hours to process for IDs: {TARGET_IDS if TARGET_IDS else 'ALL'}")
display(df_to_process[['pyroCb_id', 'time', 'pixel_latitude', 'pixel_longitude']].head())

Found 227 specific rows/hours to process for IDs: ['179', '180', '181', '189', '190', '202', '216', '253', '258', '260']


,pyroCb_id,time,pixel_latitude,pixel_longitude
0,179,5/27/2021 6:00,33.287442,-108.588483
1,179,5/27/2021 12:00,33.287442,-108.588483
2,179,5/27/2021 18:00,33.287442,-108.588483
3,179,5/28/2021 0:00,33.287442,-108.588483
4,179,5/28/2021 6:00,33.287442,-108.588483


In [ ]:
import cdsapi
import xarray as xr
import math
import numpy as np
import zipfile
import tempfile
import os
import pandas as pd # Ensure pandas is imported as it's used for isna() and to_datetime
from google.colab import userdata

# Initialize CDS Client
CDSAPI_KEY = userdata.get('CDSAPI_KEY')
c = cdsapi.Client(url='https://cds.climate.copernicus.eu/api', key=CDSAPI_KEY)

def open_era5_file(file_path):
    """Helper to open GRIB file, handling ZIP archives if necessary."""
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return None

    if zipfile.is_zipfile(file_path):
        print(f"Detected ZIP archive: {file_path}. Extracting...")
        tmpdir = tempfile.mkdtemp()
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(tmpdir)

        extracted = [os.path.join(tmpdir, f) for f in os.listdir(tmpdir) if f.endswith(('.grib', '.grb'))]
        if extracted:
            # Opening using the cfgrib engine explicitly
            datasets = [xr.open_dataset(f, engine='cfgrib') for f in extracted]
            return xr.merge(datasets) if len(datasets) > 1 else datasets[0]
    else:
        return xr.open_dataset(file_path, engine='cfgrib')

def process_era5_pressure_levels(row):
    pyro_id = str(row['pyroCb_id'])
    ts = pd.to_datetime(row['time'])
    date_str = ts.strftime('%Y-%m-%d')
    hour = ts.hour

    lat = row['pixel_latitude']
    lon = row['pixel_longitude']

    # --- NEW: Handle NaN latitudes/longitudes ---
    if pd.isna(lat) or pd.isna(lon):
        print(f"[WARNING] Skipping pyroCb_id {pyro_id} at {ts} due to NaN latitude or longitude.")
        return None
    # --- END NEW ---

    # Setup Directory Structure
    event_dir = os.path.join(OUTPUT_BASE_DIR, pyro_id)
    os.makedirs(event_dir, exist_ok=True)
    grib_path = os.path.join(event_dir, f"ERA5_PL_{pyro_id}_{date_str}_{hour:02d}UTC.grib")

    # Area selection
    delta_lat = 100 / 111.0
    delta_lon = 100 / (111.0 * math.cos(math.radians(lat)))
    area = [
        round((lat + delta_lat)*4)/4, round((lon - delta_lon)*4)/4,
        round((lat - delta_lat)*4)/4, round((lon + delta_lon)*4)/4
    ]

    # Download
    if not os.path.exists(grib_path):
        print(f"[LOG] Downloading PL data for ID {pyro_id} at {ts}...")
        try:
            c.retrieve('reanalysis-era5-pressure-levels', {
                'product_type': 'reanalysis',
                'format': 'grib',
                'variable': ['relative_humidity', 'u_component_of_wind', 'v_component_of_wind'],
                'pressure_level': ['250', '650', '750', '850'],
                'year': str(ts.year),
                'month': f"{ts.month:02d}",
                'day': f"{ts.day:02d}",
                'time': f"{hour:02d}:00",
                'area': area,
            }, grib_path)
        except Exception as e:
            print(f"[ERROR] Download failed for {pyro_id} at {ts}: {e}")
            return None

    # Extract point data using the ZIP-aware helper
    try:
        ds = open_era5_file(grib_path)
        if ds is None: return None

        point = ds.sel(latitude=lat, longitude=lon, method='nearest')

        res = {
            'pyroCb_id': row['pyroCb_id'],
            'time': row['time'],
            'pixel_latitude': row['pixel_latitude'],
            'pixel_longitude': row['pixel_longitude'],
            'rh_650': point.sel(isobaricInhPa=650).r.values.item(),
            'rh_750': point.sel(isobaricInhPa=750).r.values.item(),
            'rh_850': point.sel(isobaricInhPa=850).r.values.item(),
            'u_250': point.sel(isobaricInhPa=250).u.values.item(),
            'v_250': point.sel(isobaricInhPa=250).v.values.item()
        }
        ds.close()
        return res
    except Exception as e:
        print(f"[ERROR] Extraction failed for {pyro_id} at {ts}: {e}")
        return None

# Execute Batch
results_list = []
for index, row in df_to_process.iterrows():
    extracted = process_era5_pressure_levels(row)
    if extracted:
        results_list.append(extracted)

df_era5_new = pd.DataFrame(results_list)
print(f"Extraction complete for {len(df_era5_new)} rows.")

[WARNING] Skipping pyroCb_id 189 at 2021-06-29 19:00:00 due to NaN latitude or longitude.
[LOG] Downloading PL data for ID 189 at 2021-06-30 01:00:00...


2026-07-30 04:08:37,205 INFO Request ID is 87636785-a692-4ee5-b905-1c76dcbcdd3e
INFO:ecmwf.datastores.legacy_client:Request ID is 87636785-a692-4ee5-b905-1c76dcbcdd3e
2026-07-30 04:08:37,946 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-30 04:09:00,256 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-07-30 04:09:11,797 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


9a22b90ba0106817c489c3b5ba651efe.grib:   0%|          | 0.00/3.14k [00:00<?, ?B/s]

[LOG] Downloading PL data for ID 189 at 2021-06-30 07:00:00...


2026-07-30 04:09:13,518 INFO Request ID is 176829e8-40b2-4324-8c52-20e7f9450101
INFO:ecmwf.datastores.legacy_client:Request ID is 176829e8-40b2-4324-8c52-20e7f9450101
2026-07-30 04:09:13,665 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-30 04:09:47,935 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


...

fea2c53f7e54a01c49070dd85c4d731f.grib:   0%|          | 0.00/4.45k [00:00<?, ?B/s]

[LOG] Downloading PL data for ID 260 at 2022-06-14 02:00:00...


2026-07-30 05:40:28,732 INFO Request ID is 8c4c9847-50d6-42c2-8134-90d29e2be70b
INFO:ecmwf.datastores.legacy_client:Request ID is 8c4c9847-50d6-42c2-8134-90d29e2be70b
2026-07-30 05:40:28,887 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-07-30 05:40:50,526 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-07-30 05:41:02,064 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


38c7944bcd1b86ddb4d2984f92fbe88c.grib:   0%|          | 0.00/4.45k [00:00<?, ?B/s]

Extraction complete for 225 rows.


In [ ]:
# --- MERGE AND CLEAN ---
# Merge using 'time' (the original timestamp column in the input CSV)
merged_df = pd.merge(
    df_input,
    df_era5_new,
    on=['pyroCb_id', 'time', 'pixel_latitude', 'pixel_longitude'],
    how='left'
)

# Remove duplicate inputs if any exist based on the core unique identifiers
initial_count = len(merged_df)
merged_df = merged_df.drop_duplicates(subset=['pyroCb_id', 'time'])
final_count = len(merged_df)

print(f"Merged data. Removed {initial_count - final_count} duplicates.")

# Save result
merged_df.to_csv(FINAL_MERGED_CSV, index=False)
print(f"Final file saved to: {FINAL_MERGED_CSV}")
display(merged_df.head())

Merged data. Removed 0 duplicates.
Final file saved to: /content/drive/MyDrive/Pyrocb_data/merged_with_era5_pressure_final.csv


,pyroCb_id,time,pixel_latitude,pixel_longitude,latitude_vegetation,longitude_vegetation,cvh,cvl,tvh,tvl,...,slope,aspect_sin,aspect_cos,tpi,tri,rh_650,rh_750,rh_850,u_250,v_250
0,179,5/27/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,29.0,0.374607,0.927184,4.567567,1.0,21.704834,13.238544,16.766167,30.144974,18.562988
1,179,5/27/2021 12:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,29.0,0.374607,0.927184,4.567567,1.0,27.561798,16.804243,29.028404,31.931488,23.416809
2,179,5/27/2021 18:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,29.0,0.374607,0.927184,4.567567,1.0,27.393448,14.195095,11.276022,31.079453,21.675842
3,179,5/28/2021 0:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,29.0,0.374607,0.927184,4.567567,1.0,22.148581,12.047037,10.058139,21.766632,8.594025
4,179,5/28/2021 6:00,33.287442,-108.588483,33.25,-108.5,0.888824,0.093579,3,16,...,29.0,0.374607,0.927184,4.567567,1.0,15.003980,13.651347,16.103186,23.590179,-3.886917
